In [1]:
import os
import numpy as np
import pickle
import matplotlib.pyplot as plt

from time import time
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from collections import OrderedDict

# Load custom modules
from common.functions import *
from common.gradient import *
from common.layers import *
from common.multi_layer_net_extend import MultiLayerNetExtend
from common.multi_layer_net import MultiLayerNet
from common.optimizer import *
from common.trainer import Trainer
from common.util import *

# 한글 폰트 및 마이너스 기호 표시 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [3]:
torch.cuda.init()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.cuda.reset_peak_memory_stats(device=None)
print("현재 디바이스:", device)

os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
os.environ['CUDA_VISIBLE_DEVICES'] = "0"
os.environ['TORCH_USE_CUDA_DSA'] = "1"

현재 디바이스: cuda


In [4]:
# CIFAR-100 데이터 로드 함수
def unpickle(file):
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

data_path = './cifar-100-python'
# 데이터 경로 설정
train_path = os.path.join(data_path, 'train')
test_path = os.path.join(data_path, 'test')

# 학습 데이터 로드
train_data = unpickle(train_path)
test_data = unpickle(test_path)

# 메타데이터 로드 (클래스 이름 등)
meta_path = os.path.join(data_path, 'meta')
meta_data = unpickle(meta_path)

fine_label_names = [name.decode() for name in meta_data[b'fine_label_names']]
coarse_label_names = [name.decode() for name in meta_data[b'coarse_label_names']]

# 데이터 구조 확인
print("학습 데이터 키:", [key.decode() if isinstance(key, bytes) else key for key in train_data.keys()])
print("테스트 데이터 키:", [key.decode() if isinstance(key, bytes) else key for key in test_data.keys()])
print("메타 데이터 키:", [key.decode() if isinstance(key, bytes) else key for key in meta_data.keys()])

학습 데이터 키: ['filenames', 'batch_label', 'fine_labels', 'coarse_labels', 'data']
테스트 데이터 키: ['filenames', 'batch_label', 'fine_labels', 'coarse_labels', 'data']
메타 데이터 키: ['fine_label_names', 'coarse_label_names']


In [5]:
def one_hot_encode(y, num_classes):
    return np.eye(num_classes)[y.astype(int)]

# train set 으로 validation set 분할
x = train_data[b'data']
x = x.reshape(-1, 3, 32, 32)
t = np.array(train_data[b'coarse_labels'])
x_train, x_val, y_train, y_val = train_test_split(x, t, test_size=0.2, random_state=42, stratify=t)

# test set 정의
x_test = test_data[b'data']
x_test = x_test.reshape(-1, 3, 32, 32)
y_test = np.array(test_data[b'coarse_labels'])

x_test = x_test.astype(np.float32) / 255.0
x_train = x_train.astype(np.float32) / 255.0
x_val = x_val.astype(np.float32) / 255.0

y_train = one_hot_encode(y_train, 20)
y_val = one_hot_encode(y_val, 20)
y_test = one_hot_encode(y_test, 20)

x_train.shape, x_val.shape, x_test.shape, y_train.shape, y_val.shape, y_test.shape

((40000, 3, 32, 32),
 (10000, 3, 32, 32),
 (10000, 3, 32, 32),
 (40000, 20),
 (10000, 20),
 (10000, 20))

In [6]:
class CIFAR100Dataset(Dataset):
    def __init__(self, images, labels, transform=None): # transform 인자 추가
        self.images = images
        self.labels = labels
        self.transform = transform # transform 저장

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx] # (C, H, W) 형태의 NumPy 배열
        label = self.labels[idx]
        
        # NumPy 배열을 PyTorch 텐서로 변환
        # 이미지는 이미 float32로 정규화되어 있음
        image_tensor = torch.tensor(image) 
        label_tensor = torch.tensor(label) # 레이블은 원-핫 인코딩된 상태
        
        if self.transform:
            image_tensor = self.transform(image_tensor) # 변환 적용
            
        return image_tensor, label_tensor

In [7]:
train_transforms = transforms.Compose([
    transforms.RandomCrop(32, padding=4),  # 이미지 주위에 4픽셀 패딩 후 32x32 랜덤 크롭
    transforms.RandomHorizontalFlip(p=0.5), # 50% 확률로 좌우 반전
    # 필요한 경우 다른 변환 추가 가능 (예: transforms.ColorJitter)
])
train_dataset = CIFAR100Dataset(x_train, y_train, transform=train_transforms)
val_dataset = CIFAR100Dataset(x_val, y_val)
test_dataset = CIFAR100Dataset(x_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [8]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # Conv1: Input (3, 32, 32) -> Output (32, 16, 16)
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1, stride=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Conv2: Input (32, 16, 16) -> Output (64, 8, 8)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1, stride=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Conv3: Input (64, 8, 8) -> Output (128, 8, 8)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1, stride=1)
        self.relu3 = nn.ReLU()
        # No pooling after conv3

        # Calculate the flattened size after conv3
        # Output of conv3 will be (batch_size, 128, 8, 8)
        self.flattened_size = 128 * 8 * 8

        # Affine1 (FC1)
        self.fc1 = nn.Linear(self.flattened_size, 128)
        self.relu4 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.5) # 과적합 방지를 위해 드롭아웃 추가 고려

        # Affine2 (FC2 - Output layer)
        self.fc2 = nn.Linear(128, 20) # 20 coarse labels

    def forward(self, x):
        # Conv block 1
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        # Conv block 2
        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        # Conv block 3
        x = self.conv3(x)
        x = self.relu3(x)

        # Flatten
        x = x.view(-1, self.flattened_size)

        # FC block 1
        x = self.fc1(x)
        x = self.relu4(x)
        x = self.dropout1(x) # 드롭아웃 적용
        
        # FC block 2 (Output)
        x = self.fc2(x)
        return x

In [9]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        
        # Convolutional Block 1 (Inspired by Keras example)
        self.conv1_1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1) # padding='same'
        self.relu1_1 = nn.ReLU()
        self.conv1_2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1) # padding='same'
        self.relu1_2 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2) # (32, 16, 16)

        # Convolutional Block 2 (Inspired by Keras example)
        self.conv2_1 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1) # padding='same'
        self.relu2_1 = nn.ReLU()
        self.conv2_2 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1) # padding='same'
        self.relu2_2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2) # (128, 8, 8)
        
        # Note: Keras example has one more Conv2D(128, (3,3)) after pooling, 
        # but for 20 classes and to keep it slightly simpler, we'll go to FC layers.
        # If needed, that layer can be added:
        # self.conv3_1 = nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, padding=1)
        # self.relu3_1 = nn.ReLU()
        # self.flattened_size = 128 * 8 * 8 (if conv3_1 is added and no pooling after)

        self.flattened_size = 128 * 8 * 8 # 8192
        
        # Dense Block 1
        self.fc1 = nn.Linear(self.flattened_size, 256)
        self.relu_fc1 = nn.ReLU()
        self.bn_fc1 = nn.BatchNorm1d(256)
        self.drop_fc1 = nn.Dropout(p=0.3)

        # Dense Block 2
        self.fc2 = nn.Linear(256, 256)
        self.relu_fc2 = nn.ReLU()
        self.bn_fc2 = nn.BatchNorm1d(256)
        self.drop_fc2 = nn.Dropout(p=0.3)

        # Output Layer (for 20 coarse labels)
        self.fc_out = nn.Linear(256, 20)

    def forward(self, x):
        # Conv Block 1
        x = self.relu1_1(self.conv1_1(x))
        x = self.relu1_2(self.conv1_2(x))
        x = self.pool1(x)
        
        # Conv Block 2
        x = self.relu2_1(self.conv2_1(x))
        x = self.relu2_2(self.conv2_2(x))
        x = self.pool2(x)
        
        # Flatten
        x = x.view(-1, self.flattened_size)
        
        # Dense Block 1
        x = self.fc1(x)
        x = self.relu_fc1(x)
        x = self.bn_fc1(x)
        x = self.drop_fc1(x)
        
        # Dense Block 2
        x = self.fc2(x)
        x = self.relu_fc2(x)
        x = self.bn_fc2(x)
        x = self.drop_fc2(x)
        
        # Output Layer
        x = self.fc_out(x)
        return x

In [10]:
model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [12]:
num_epochs = 100

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        target_labels = torch.max(labels, 1)[1] 
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, target_labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        _, predicted_train = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted_train == target_labels).sum().item()

    epoch_train_loss = running_loss / total_train
    epoch_train_acc = 100 * correct_train / total_train
    
    # --- 검증 단계 ---
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            target_labels = torch.max(labels, 1)[1]
            
            outputs = model(images)
            loss = criterion(outputs, target_labels)
            val_loss += loss.item() * images.size(0)
            
            _, predicted_val = torch.max(outputs.data, 1)
            total_val += labels.size(0)
            correct_val += (predicted_val == target_labels).sum().item()
            
    epoch_val_loss = val_loss / total_val
    epoch_val_acc = 100 * correct_val / total_val
    
    print(f'Epoch [{epoch+1}/{num_epochs}], '
          f'Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.2f}%, '
          f'Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.2f}%')

# --- 최종 테스트 단계 ---
model.eval()
correct_test = 0
total_test = 0
with torch.no_grad():
    for images, labels in test_loader: # test_loader 사용
        images, labels = images.to(device), labels.to(device)
        target_labels = torch.max(labels, 1)[1]
        
        outputs = model(images)
        _, predicted_test = torch.max(outputs.data, 1)
        total_test += labels.size(0)
        correct_test += (predicted_test == target_labels).sum().item()

final_test_acc = 100 * correct_test / total_test
print(f'Test Accuracy on {total_test} test images: {final_test_acc:.2f}%')

Epoch [1/100], Train Loss: 1.0593, Train Acc: 65.96%, Val Loss: 1.0524, Val Acc: 66.43%
Epoch [2/100], Train Loss: 1.0491, Train Acc: 66.14%, Val Loss: 1.1486, Val Acc: 63.95%
Epoch [3/100], Train Loss: 1.0496, Train Acc: 66.00%, Val Loss: 1.0649, Val Acc: 66.47%
Epoch [4/100], Train Loss: 1.0448, Train Acc: 66.26%, Val Loss: 1.0418, Val Acc: 66.54%
Epoch [5/100], Train Loss: 1.0412, Train Acc: 66.14%, Val Loss: 1.0207, Val Acc: 67.47%
Epoch [6/100], Train Loss: 1.0408, Train Acc: 66.44%, Val Loss: 1.0627, Val Acc: 65.81%
Epoch [7/100], Train Loss: 1.0317, Train Acc: 66.58%, Val Loss: 1.0520, Val Acc: 66.18%
Epoch [8/100], Train Loss: 1.0252, Train Acc: 66.69%, Val Loss: 1.0402, Val Acc: 66.93%
Epoch [9/100], Train Loss: 1.0206, Train Acc: 66.77%, Val Loss: 1.1239, Val Acc: 64.63%
Epoch [10/100], Train Loss: 1.0112, Train Acc: 67.43%, Val Loss: 1.0793, Val Acc: 66.18%
Epoch [11/100], Train Loss: 1.0080, Train Acc: 67.35%, Val Loss: 1.0474, Val Acc: 66.83%
Epoch [12/100], Train Loss: 0.